# Phase 2 - Preprocessing

Ce notebook formalise les decisions de preprocessing issues de l'EDA. Certaines transformations peuvent etre appliquees directement sans fuite de donnees, tandis que d'autres doivent etre integrees plus tard dans un pipeline `sklearn` apres la separation train/test.

## 1. Setup et chargement des donnees

In [17]:
# Import des bibliotheques necessaires.
from pathlib import Path

import numpy as np
import pandas as pd
import os

# Chargement du dataset depuis la racine du projet ou depuis le dossier notebooks.
current_path = os.getcwd()
data_path = data_path = Path(current_path) / 'modeling' / 'data' / 'dataset.csv'
if not data_path.exists():
    data_path = Path('data') / 'dataset.csv'

df = pd.read_csv(data_path)

print('Shape initiale:', df.shape)
display(df.head())

Shape initiale: (10603, 24)


,country,year,current_account_pct,debt_service_pct,domestic_credit_pct,external_debt_pct,fdi_inflows,fx_reserves_months,gdp_growth,gdp_per_capita,...,trade_openness,unemployment,country_name,region,income_group,lending_type,gdp_growth_lag1,gdp_growth_delta,is_crisis_decade,gdp_shock
0,ABW,1988,-7.425094,NaN,NaN,NaN,2.055486,3.384498,18.648649,22468.507120,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,16.078431,2.570217,1980s,0
1,ABW,1989,-6.714859,NaN,NaN,NaN,0.186908,2.921822,12.129841,24730.396448,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,18.648649,-6.518808,1980s,0
2,ABW,1990,-20.679328,NaN,NaN,NaN,17.063550,2.217400,3.961402,24763.653810,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,12.129841,-8.168439,1990s,0
3,ABW,1991,-24.023062,NaN,NaN,NaN,21.185138,1.179546,7.962872,25460.363027,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,3.961402,4.001470,1990s,0
4,ABW,1992,4.568765,NaN,NaN,NaN,-3.857809,1.292875,5.882354,25743.445501,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,7.962872,-2.080518,1990s,0


## 2. Suppression des colonnes avec trop de valeurs manquantes

Decision: supprimer entierement `domestic_credit_pct` et `govt_debt_pct_gdp`.

- `domestic_credit_pct`: 86.75% de valeurs manquantes.
- `govt_debt_pct_gdp`: 85.16% de valeurs manquantes.

Ces taux sont trop eleves pour une imputation fiable. Garder ces variables ajouterait beaucoup de bruit et reduirait la robustesse du modele.

In [18]:
# Suppression des colonnes jugees inutilisables a cause du taux de valeurs manquantes.
high_missing_cols = ['domestic_credit_pct', 'govt_debt_pct_gdp']

df_preprocessed = df.drop(columns=high_missing_cols)

print('Colonnes supprimees:', high_missing_cols)
print('Shape apres suppression des colonnes:', df_preprocessed.shape)

Colonnes supprimees: ['domestic_credit_pct', 'govt_debt_pct_gdp']
Shape apres suppression des colonnes: (10603, 22)


## 3. Gestion des valeurs manquantes restantes

Decision appliquee directement: supprimer les lignes ou `gdp_per_capita` est manquant, car le taux de valeurs manquantes est tres faible (0.77%).

Decision a appliquer plus tard dans le pipeline: pour les variables avec 20-60% de valeurs manquantes, utiliser une imputation par mediane groupee par `income_group`.

Variables concernees:

- `gov_expenditure`
- `debt_service_pct`
- `external_debt_pct`
- `gni_per_capita_growth`
- `unemployment`
- `fx_reserves_months`
- `current_account_pct`
- `trade_openness`
- `inflation`
- `fdi_inflows`

Justification: les pays d'un meme groupe de revenu ont des profils economiques plus proches. Une mediane par `income_group` est donc plus adaptee qu'une mediane globale.

**Important:** les statistiques d'imputation doivent etre calculees uniquement sur le train set. Elles ne doivent pas etre calculees sur le dataset complet, afin d'eviter une fuite de donnees. La separation train/test n'etant pas encore faite, cette etape est seulement definie ici, pas ajustee.

In [19]:
# Suppression directe des lignes ou gdp_per_capita est manquant.
rows_before = len(df_preprocessed)
df_preprocessed = df_preprocessed.dropna(subset=['gdp_per_capita']).copy()
rows_after = len(df_preprocessed)

print('Lignes supprimees pour gdp_per_capita manquant:', rows_before - rows_after)
print('Shape apres suppression des lignes:', df_preprocessed.shape)

Lignes supprimees pour gdp_per_capita manquant: 82
Shape apres suppression des lignes: (10521, 22)


In [20]:
# Definition de la strategie d'imputation, sans calculer les medianes sur le dataset complet.
# Les medianes devront etre apprises uniquement sur le train set apres split.
income_group_median_impute_cols = [
    'gov_expenditure', 'debt_service_pct', 'external_debt_pct',
    'gni_per_capita_growth', 'unemployment', 'fx_reserves_months',
    'current_account_pct', 'trade_openness', 'inflation', 'fdi_inflows'
]

imputation_strategy = {
    'group_column': 'income_group',
    'method': 'median_by_group',
    'fit_on': 'train_only',
    'columns': income_group_median_impute_cols,
}

imputation_strategy

{'group_column': 'income_group',
 'method': 'median_by_group',
 'fit_on': 'train_only',
 'columns': ['gov_expenditure',
  'debt_service_pct',
  'external_debt_pct',
  'gni_per_capita_growth',
  'unemployment',
  'fx_reserves_months',
  'current_account_pct',
  'trade_openness',
  'inflation',
  'fdi_inflows']}

## 4. Feature engineering

Deux nouvelles variables sont ajoutees:

- `gdp_per_capita_log = np.log(gdp_per_capita)`: transformation logarithmique pour reduire les differences d'echelle entre pays. Le PIB par habitant varie fortement entre pays, donc le log rend les ecarts plus proportionnels economiquement.
- `high_external_debt = 1 si external_debt_pct > 80, sinon 0`: indicateur binaire de vulnerabilite liee a la dette externe. Les valeurs manquantes de `external_debt_pct` sont traitees comme 0 pour ce flag.

In [21]:
# Transformation logarithmique du PIB par habitant.
df_preprocessed['gdp_per_capita_log'] = np.log(df_preprocessed['gdp_per_capita'])

# Creation d'un flag de dette externe elevee.
# Les NaN donnent False dans la comparaison, donc le flag vaut 0 si la valeur est manquante.
df_preprocessed['high_external_debt'] = (df_preprocessed['external_debt_pct'] > 80).astype(int)

display(df_preprocessed[['gdp_per_capita', 'gdp_per_capita_log', 'external_debt_pct', 'high_external_debt']].head())

,gdp_per_capita,gdp_per_capita_log,external_debt_pct,high_external_debt
0,22468.507120,10.019870,NaN,0
1,24730.396448,10.115788,NaN,0
2,24763.653810,10.117132,NaN,0
3,25460.363027,10.144878,NaN,0
4,25743.445501,10.155935,NaN,0


## 5. Decision de fuite de donnees

Decision equipe: supprimer `gdp_growth`, `gdp_growth_lag1` et `gdp_growth_delta` de la matrice de variables explicatives. Ces trois variables ont ete utilisees directement pour construire la cible `gdp_shock`. Les conserver reviendrait a permettre au modele de retrouver la formule de labellisation, au lieu dapprendre des signaux economiques generaux.

Cette decision reduira probablement les performances apparentes en Phase 3, mais elle rendra levaluation plus honnete et plus credible.


In [22]:
# Suppression des variables utilisees pour construire la cible afin deviter la fuite de donnees.
leakage_cols = ['gdp_growth', 'gdp_growth_lag1', 'gdp_growth_delta']

df_preprocessed = df_preprocessed.drop(columns=leakage_cols)

print('Colonnes supprimees pour fuite de donnees:', leakage_cols)
print('Shape apres suppression des variables de fuite:', df_preprocessed.shape)


Colonnes supprimees pour fuite de donnees: ['gdp_growth', 'gdp_growth_lag1', 'gdp_growth_delta']
Shape apres suppression des variables de fuite: (10521, 21)


## 6. Decisions d'encodage categoriel

Ces encodages ne sont pas appliques dans ce notebook. Ils seront integres plus tard dans le pipeline `sklearn`.

- `region`: One-Hot Encoding, car la variable est nominale et contient 7 modalites.
- `income_group`: One-Hot Encoding, car les ecarts entre niveaux de revenu ne sont pas garantis comme egaux.
- `lending_type`: One-Hot Encoding, car la variable est nominale.
- `is_crisis_decade`: Ordinal Encoding, car il existe un ordre temporel clair: `1960s < 1970s < 1980s < 1990s < 2000s < 2010s < 2020s`.

## 7. Strategie pour les outliers

Les valeurs extremes observees ne sont pas considerees comme des erreurs:

- `current_account_pct` varie environ de -148 a +312.
- `gdp_growth_delta` varie environ de -126 a +137.

Decision: conserver ces valeurs, car elles representent des extremes economiques reels.

Choix du scaler: utiliser `RobustScaler` pour les variables numeriques. Ce scaler repose sur la mediane et l'IQR, il est donc plus resistant aux outliers. `StandardScaler` serait davantage influence par ces valeurs extremes.

## 8. Decision confirmee: risque de fuite de donnees

`gdp_growth`, `gdp_growth_lag1` et `gdp_growth_delta` sont exclus de la matrice de features. Cette decision est appliquee dans la section 5, car ces variables ont servi a definir `gdp_shock`.

## 9. Apercu du dataset apres les transformations confirmees

In [23]:
# Verification finale des transformations appliquees directement.
print('Shape finale provisoire:', df_preprocessed.shape)
display(df_preprocessed.head())
display(df_preprocessed.dtypes)

Shape finale provisoire: (10521, 21)


,country,year,current_account_pct,debt_service_pct,external_debt_pct,fdi_inflows,fx_reserves_months,gdp_per_capita,gni_per_capita_growth,gov_expenditure,...,trade_openness,unemployment,country_name,region,income_group,lending_type,is_crisis_decade,gdp_shock,gdp_per_capita_log,high_external_debt
0,ABW,1988,-7.425094,NaN,NaN,2.055486,3.384498,22468.507120,NaN,NaN,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,1980s,0,10.019870,0
1,ABW,1989,-6.714859,NaN,NaN,0.186908,2.921822,24730.396448,NaN,NaN,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,1980s,0,10.115788,0
2,ABW,1990,-20.679328,NaN,NaN,17.063550,2.217400,24763.653810,NaN,NaN,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,1990s,0,10.117132,0
3,ABW,1991,-24.023062,NaN,NaN,21.185138,1.179546,25460.363027,NaN,NaN,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,1990s,0,10.144878,0
4,ABW,1992,4.568765,NaN,NaN,-3.857809,1.292875,25743.445501,NaN,NaN,...,NaN,NaN,Aruba,Latin America & Caribbean,High income,Not classified,1990s,0,10.155935,0


country                      str
year                       int64
current_account_pct      float64
debt_service_pct         float64
external_debt_pct        float64
fdi_inflows              float64
fx_reserves_months       float64
gdp_per_capita           float64
gni_per_capita_growth    float64
gov_expenditure          float64
inflation                float64
trade_openness           float64
unemployment             float64
country_name                 str
region                       str
income_group                 str
lending_type                 str
is_crisis_decade             str
gdp_shock                  int64
gdp_per_capita_log       float64
high_external_debt         int64
dtype: object

## 10. Pipeline de preprocessing final

Cette section construit les jeux train, validation et test, puis ajuste le pipeline de preprocessing uniquement sur le train set. Aucun modele nest ajoute ici: la modelisation sera faite en Phase 3.


### Etape 1 - Separation train / validation / test

On separe dabord les variables explicatives `X` et la cible `y`. Les colonnes `country`, `country_name` et `year` sont exclues de `X`, car ce sont des identifiants et non des variables explicatives. La separation est stratifiee sur `gdp_shock` pour conserver le ratio de classe minoritaire dans chaque sous-ensemble.


In [24]:
from sklearn.model_selection import train_test_split

target_col = 'gdp_shock'
identifier_cols = ['country', 'country_name', 'year']

X = df_preprocessed.drop(columns=identifier_cols + [target_col])
y = df_preprocessed[target_col].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

def print_split_summary(name, X_split, y_split):
    distribution = (
        y_split.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
    counts = y_split.value_counts().sort_index()
    print(f'{name}: X={X_split.shape}, y={y_split.shape}')
    print('Distribution gdp_shock - counts:')
    print(counts)
    print('Distribution gdp_shock - pct:')
    print(distribution)
    print('-' * 50)

print('Verification des splits stratifies')
print_split_summary('Train', X_train, y_train)
print_split_summary('Validation', X_val, y_val)
print_split_summary('Test', X_test, y_test)


Verification des splits stratifies
Train: X=(7364, 17), y=(7364,)
Distribution gdp_shock - counts:
gdp_shock
0    6754
1     610
Name: count, dtype: int64
Distribution gdp_shock - pct:
gdp_shock
0    91.72
1     8.28
Name: proportion, dtype: float64
--------------------------------------------------
Validation: X=(1578, 17), y=(1578,)
Distribution gdp_shock - counts:
gdp_shock
0    1448
1     130
Name: count, dtype: int64
Distribution gdp_shock - pct:
gdp_shock
0    91.76
1     8.24
Name: proportion, dtype: float64
--------------------------------------------------
Test: X=(1579, 17), y=(1579,)
Distribution gdp_shock - counts:
gdp_shock
0    1448
1     131
Name: count, dtype: int64
Distribution gdp_shock - pct:
gdp_shock
0    91.7
1     8.3
Name: proportion, dtype: float64
--------------------------------------------------


### Etape 2 - Definition du ColumnTransformer

Les listes de colonnes sont construites a partir des colonnes effectivement presentes dans `df_preprocessed`. Les variables numeriques continues sont imputees par mediane puis transformees avec `RobustScaler`. Les variables categorielles sont imputees par modalite la plus frequente avant encodage. `high_external_debt` et `gdp_per_capita_log` sont conservees sans transformation supplementaire.


In [25]:
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

from modeling.src.preprocessing import GroupMedianImputer

nominal_categorical_cols = ['region', 'income_group', 'lending_type']
ordinal_categorical_cols = ['is_crisis_decade']
binary_passthrough_cols = ['high_external_debt']
log_passthrough_cols = ['gdp_per_capita_log']
passthrough_cols = binary_passthrough_cols + log_passthrough_cols

excluded_numeric_cols = ['gdp_per_capita', 'gdp_per_capita_log']
numeric_cols = [
    col
    for col in X_train.select_dtypes(include=['float']).columns
    if col not in excluded_numeric_cols
]
numeric_imputer_input_cols = numeric_cols + ['income_group']

expected_cols = numeric_cols + nominal_categorical_cols + ordinal_categorical_cols + passthrough_cols
missing_expected_cols = [col for col in expected_cols if col not in X_train.columns]
if missing_expected_cols:
    raise ValueError(f'Colonnes attendues absentes de X_train: {missing_expected_cols}')

dropped_by_remainder = sorted(set(X_train.columns) - set(expected_cols))

numeric_pipeline = Pipeline(steps=[
    ('imputer', GroupMedianImputer(group_col='income_group', feature_cols=numeric_cols)),
    ('scaler', RobustScaler()),
])

nominal_categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

ordinal_categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']])),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_imputer_input_cols),
        ('nominal_categorical', nominal_categorical_pipeline, nominal_categorical_cols),
        ('ordinal_categorical', ordinal_categorical_pipeline, ordinal_categorical_cols),
        ('passthrough', 'passthrough', passthrough_cols),
    ],
    remainder='drop',
)

print('Colonnes numeriques imputees par mediane income_group + scalees:', numeric_cols)
print('Colonne de groupe pour imputation numerique:', 'income_group')
print('Colonnes categorielles one-hot encodees:', nominal_categorical_cols)
print('Colonnes categorielles ordinales:', ordinal_categorical_cols)
print('Colonnes en passthrough:', passthrough_cols)
print('Colonnes ignorees par remainder=drop:', dropped_by_remainder)


Colonnes numeriques imputees par mediane income_group + scalees: ['current_account_pct', 'debt_service_pct', 'external_debt_pct', 'fdi_inflows', 'fx_reserves_months', 'gni_per_capita_growth', 'gov_expenditure', 'inflation', 'trade_openness', 'unemployment']
Colonne de groupe pour imputation numerique: income_group
Colonnes categorielles one-hot encodees: ['region', 'income_group', 'lending_type']
Colonnes categorielles ordinales: ['is_crisis_decade']
Colonnes en passthrough: ['high_external_debt', 'gdp_per_capita_log']
Colonnes ignorees par remainder=drop: ['gdp_per_capita']


### Etape 3 - Construction du pipeline complet

Le `ColumnTransformer` est encapsule dans un `sklearn.pipeline.Pipeline` avec une seule etape nommee `preprocessor`. Aucun estimateur nest ajoute a ce stade.


In [26]:
preprocessing_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
])

print(preprocessing_pipeline)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   GroupMedianImputer(feature_cols=['current_account_pct',
                                                                                                    'debt_service_pct',
                                                                                                    'external_debt_pct',
                                                                                                    'fdi_inflows',
                                                                                                    'fx_reserves_months',
                                                                                                    'gni_per_capita_growth',
                                                                                                    'g

### Etape 4 - Ajustement sur le train set et transformation des trois jeux

Le pipeline est ajuste uniquement sur `X_train`. Les jeux validation et test sont seulement transformes, afin deviter toute fuite dinformation depuis les donnees devaluation.


In [27]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)
X_test_processed = preprocessing_pipeline.transform(X_test)

print('Shape X_train apres transformation:', X_train_processed.shape)
print('Shape X_val apres transformation:', X_val_processed.shape)
print('Shape X_test apres transformation:', X_test_processed.shape)


Shape X_train apres transformation: (7364, 29)
Shape X_val apres transformation: (1578, 29)
Shape X_test apres transformation: (1579, 29)


### Etape 5 - Sauvegarde des datasets transformes

Les matrices transformees sont converties en DataFrames, puis la cible `gdp_shock` est ajoutee a chaque fichier. Les fichiers sont sauvegardes dans `data/processed/`.


In [28]:
processed_dir = Path('..') / 'data' / 'processed'
if not processed_dir.parent.exists():
    processed_dir = Path('data') / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

feature_names = preprocessing_pipeline.named_steps['preprocessor'].get_feature_names_out()

train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
validation_processed_df = pd.DataFrame(X_val_processed, columns=feature_names, index=X_val.index)
test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

train_processed_df[target_col] = y_train.values
validation_processed_df[target_col] = y_val.values
test_processed_df[target_col] = y_test.values

train_path = processed_dir / 'train.csv'
validation_path = processed_dir / 'validation.csv'
test_path = processed_dir / 'test.csv'

train_processed_df.to_csv(train_path, index=False)
validation_processed_df.to_csv(validation_path, index=False)
test_processed_df.to_csv(test_path, index=False)

print('Datasets transformes sauvegardes:')
print(train_path)
print(validation_path)
print(test_path)
print('Shape train.csv:', train_processed_df.shape)
print('Shape validation.csv:', validation_processed_df.shape)
print('Shape test.csv:', test_processed_df.shape)


Datasets transformes sauvegardes:
data\processed\train.csv
data\processed\validation.csv
data\processed\test.csv
Shape train.csv: (7364, 30)
Shape validation.csv: (1578, 30)
Shape test.csv: (1579, 30)


### Etape 6 - Serialisation du pipeline

Le pipeline de preprocessing ajuste sur le train set est sauvegarde avec `joblib`. Il pourra etre recharge en Phase 3 pour garantir exactement les memes transformations.


In [29]:
import joblib

# Chargement du dataset depuis la racine du projet ou depuis le dossier notebooks.
current_path = os.getcwd()


models_dir = Path(current_path) / 'modeling' / 'data'  / 'models'
if not models_dir.parent.exists():
    models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

preprocessor_path = models_dir / 'preprocessor.joblib'
joblib.dump(preprocessing_pipeline, preprocessor_path)

print('Pipeline de preprocessing sauvegarde:')
print(preprocessor_path)


Pipeline de preprocessing sauvegarde:
C:\Users\pc\Desktop\MachineLearningProjects\WorldBank\modeling\data\models\preprocessor.joblib


### Etape 7 - Preparation des strategies de desequilibre de classes

Les strategies suivantes seront testees en Phase 3, mais ne sont pas implementees dans ce notebook:

1. **Baseline:** aucun reechantillonnage, avec `class_weight='balanced'` pour les modeles qui le supportent.
2. **Oversampling:** utilisation de `SMOTE` depuis `imbalanced-learn` pour augmenter la classe minoritaire dans le train set.
3. **Undersampling:** utilisation de `RandomUnderSampler` depuis `imbalanced-learn` pour reduire la classe majoritaire dans le train set.

Important: en Phase 3, les strategies avec reechantillonnage devront utiliser `imblearn.pipeline.Pipeline`, et non `sklearn.pipeline.Pipeline`, afin que le reechantillonnage soit applique uniquement pendant lentra?nement et jamais sur les jeux validation ou test.


### Verification finale des sorties

Cette verification confirme que les trois datasets transformes et le pipeline serialise ont bien ete crees.


In [30]:
output_files = [train_path, validation_path, test_path, preprocessor_path]
missing_outputs = [path for path in output_files if not path.exists()]

if missing_outputs:
    raise FileNotFoundError(f'Fichiers de sortie manquants: {missing_outputs}')

print('Confirmation finale: tous les fichiers de sortie ont ete crees.')
for path in output_files:
    print(f'- {path}')


Confirmation finale: tous les fichiers de sortie ont ete crees.
- data\processed\train.csv
- data\processed\validation.csv
- data\processed\test.csv
- C:\Users\pc\Desktop\MachineLearningProjects\WorldBank\modeling\data\models\preprocessor.joblib
